# V2 训练数据：候选设计与原图检索

当前先逐算子 review；编辑原图来源、循环编排仍待讨论。已知循环续跑会额外重算部分阶段，尚未定稿。线上 RAG 后续单独设计。

固定文章／视觉发布 → 统一准备构题材料 → 知识／参考／目标共同设计 →（仅编辑：搜原图、核验、定稿）→ 校验与审核 → 校验已选训练输入 → 目标检索／审核 → 导出。

**输入不再包含手写plan、intent、选材ID；候选目标由流程自动检索。**全局seed、预算、题型、模型及声明的原图检索源属于运行配置。任务由知识、条件与视觉推理支撑；简单直接的知识应用也可成立，明确执行要求只作辅助，以新题实际检查信号、题面和原判据的对应，不要求training-free提升。未发布知识不会进入模型；配图、编辑原图、监督目标分别管理。已有知识本身仍有质量限制，因此新产物称开发候选。

从下面第一个代码cell开始；默认只读实际保存结果。所有展示使用Markdown、表格和原生图片，不使用HTML。

本册每一步直接执行与整体版相同的算子；按输入、prompt、调用、输出排列。`plan`仅为选题后派生的类型／划分元数据，不能在入口手填。

训练目标候选数通过 target_candidates_per_task 配置；同一冻结题目可逐张核验多张独立目标，只有通过目标审核的记录才能导出。引用ID缺口保留审计，不提前阻止寻找目标。


**当前按知识应用与候选目标共同构题思路迭代。** 新题用于检查材料、任务、公开展示条件与原题判据的对应；失败可用于研究训练，但不等于正确监督已备齐。训练输入与题目共同选定并绑定；评测仍可独立使用公开题面检索。旧案例仍按冻结版本解释；新版执行使用新的NEW_RUN。

当前主要prompt：[候选设计](prompts/design_candidates.md)、[原图审核（本地与外搜共用）](prompts/select_edit_source.md)、[编辑定稿](prompts/construct.md)、[任务审核](prompts/review_task.md)、[训练目标审核](prompts/review_target.md)。逐步版每个模型算子前展示当前prompt全文；旧案例的实际输入以其冻结请求为准。

In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")]
PROJECT = next((p for p in _candidates if (p / "curation/training/authoring.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("找不到包含curation的项目根目录")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    from curation.preparation.inspection import show_records, show_prompt, show_summary, snapshot_ref, check_step_order
    from curation.training.runtime import config as build_config
    from curation.preparation.records import run_records, run_manifest, rows
except ModuleNotFoundError as error:
    raise RuntimeError("请选择demiwtg内核：/yzp/zhaozy/yangzepeng/0905/env/bin/python") from error

MODE = "view_saved"  # execute运行；view_saved只读已有checkpoint。
BASE = PROJECT / "curation/training/runs"
SAVED_RUN = BASE / "pipeline_v2_training_review"
NEW_RUN = PROJECT / "curation/training/runs/pipeline_v2_training_target_loop_review_20260921"  # 输入／代码／prompt变化须新run。
knowledge_runs = []  # 可空：知识文本不是强制依据。
visual_runs = []  # 独立视觉发布；两类输入不能同时为空。
# 知识输入为最终交付；编辑原图检索的全库清单及外部源在下面config中公开配置。
# V2 训练构题可看独立候选目标；目标不进入知识证据或作答输入。
MODEL_BACKEND = "offline"  # local用本地Qwen；offline为相同prompt／context生成绑定请求。
# concepts=None使用全部发布概念；可传名称列表做可复现小批。
config = build_config(MODEL_BACKEND, concepts=None, seed=0, max_units=2, tasks_per_unit=1, training_sample_goal=2,
    reference_batch_size=4, max_target_cycles=2, max_training_attempts=100, max_context_chars=100000,
    scene_search={"image_ref": None,
                  "external_providers": ["commons"]},
    author_model="gpt-6-astra" if MODEL_BACKEND == "offline" else None,
    author_effort="high" if MODEL_BACKEND == "offline" else None)
CASE_ID = None  # 默认展示本阶段第一条实际参与处理的记录；也可填unit_id或task_id。
SHOW_AUDIT = False  # 完整来源、字段、实际模型请求；图片按角色原样展示。
THROUGH = "export"
if MODE not in {"view_saved", "execute"}:
    raise ValueError("MODE必须是view_saved或execute")
run = SAVED_RUN if MODE == "view_saved" else NEW_RUN
if MODE == "view_saved":
    saved = run_records(run).get("manifest")
    if saved is not None:
        config = saved["config"]
    else:
        print("请指定已有 V2 Lance run；后续查看单元需要已完成的阶段。")
print("项目：", PROJECT, "\n内核：", sys.executable, "\n模式：", MODE, "\n运行：", run)


print("本次运行配置：", config)

print("当前入口只读取 V2 Lance 结果。历史查看册保存在归档与 reviews 中。")


## 循环规则草案（按训练样本条数）

程序按固定种子遍历目标，不按描述相关性截取 top-k。一次给模型 **一个目标＋一批参考候选**，最多形成一条训练样本。通过任务与目标审核后计 1 条，立即换下一目标；失败则看下一批材料，材料耗尽也换目标。目标池走完而样本数未够时循环回来，允许重复目标，不要求题面不同。

`training_sample_goal` 控制合格样本总数；`reference_batch_size` 控制每批参考图数；`max_target_cycles`、`max_training_attempts` 与模型调用预算限制探索量。

计数与游标都是程序变量。已有 Lance checkpoint 保存每次尝试、审核结果和样本身份，恢复时重放结果重建进度；不新增 JSON／计数文件，不让模型维护计数或历史题目列表。

下方逐算子单元展示第一张目标的一批材料。末尾“按总条数继续循环”使用主 notebook 的同一条 Dataset 链，已完成的步骤自动重放，待响应步骤不会被跳过。
**本轮 review 尚未定稿的边界：** 编辑原图来源仍沿用旧检索分支，尚未讨论怎样与先选定的目标建立可信编辑对应，不能视为已确认方案。循环当前由主 notebook 外层驱动，每次尝试内部仍是标准 Dataset 算子；demiflow 是否提供通用反馈迭代能力尚待讨论，各算子不共享可随意修改的全局状态。

**隔离回归现状：** 相关测试 22 通过、1 失败。重复目标可能复用相同构题响应，首次执行与续跑记录的响应依赖不一致，使部分 checkpoint 在续跑时重新计算；样本计数和目标轮换断言已通过，但不能宣称恢复验收通过。没有正式模型调用、数据生产或训练。


## 初始化运行

冻结知识文件、引用来源文件、配置、代码与prompt版本；检查原始像素是否被修改。正式测试保留表若存在可通过config传入；省略时只允许开发用途。

In [ ]:
from functools import partial
from demiflow.standalone import local_data
from curation.preparation.records import run_lock
from curation.training.runtime import graph_version
from curation.training.authoring import (AuthoringRunFiles, input_records, split_guard,
    knowledge_items, SelectTargetCandidate)
from curation.training.candidates import ExpandCandidates
from curation.training.scene_search import SceneSearch, SearchLocalScenes, SearchExternalScenes
from curation.training.scene_assets import ExistingImagePool
from curation.training.materials import PrepareTrainingMaterials, design_concepts
from curation.training.operators import ValidateTask, BindTrainingInputs, export_record
from curation.training.prompting import (prompt_config, prompt_responses,
    prepare_design, apply_design, prepare_external_edit_source, apply_external_edit_source,
    prepare_edit_source, apply_edit_source,
    prepare_construct, apply_construct, prepare_task_review, apply_task_review,
    prepare_target_review, apply_target_review)


from curation.training.loop import training_targets, training_attempt, loop_progress


In [ ]:
if MODE == "execute":
    with run_lock(run):
        files = AuthoringRunFiles(run, knowledge_runs, "training", config, graph_version("training"), visual_runs=visual_runs)
        guard = split_guard(files)
        pack, options = prompt_config(run, config)
        data = local_data(prompt_packs={"tasks.yaml": pack}, prompt_options=options,
                          max_prompt_requests=config["model"]["max_calls"])
        prefix, task_prefix = "", None
else:
    prefix, task_prefix = "", None
    data = local_data()
    saved_manifest = run_manifest(run)
    print("当前展示的冻结输入：", saved_manifest["knowledge_runs"])


## 1. 准备构题材料（合并入口，零模型调用）

`map(PrepareTrainingMaterials)` 一次交付材料包，不再单设 ReadKnowledge checkpoint。

| 部分 | 作用与边界 |
|---|---|
| 可选知识文本 | 来自固定文章发布；构题不强制引用 |
| 可用视觉材料 | 文章配图及独立视觉发布；保留支持范围、区域和限制，未被文章采用也可用 |
| 候选目标 | 从声明发布携带的图片读取；独立视觉已做概念审核，其他原始候选不能自动视为审核通过；不扫描全 raw |
| 逐目标参考列表 | `target_reference_options` 保留原始资料编号；仅排除目标／近重复与依赖被排除配图的文字，不按指纹二分用途 |
| 预算与缺口 | seed、concepts、max_units 确定探索概念；全量材料保留审计，未选择的材料不会进入这道题训练输入。目标数量是上限，空参考列表保留给构题器判断 |

输出 `materials`、`design_targets`、`target_reference_options`、来源、审核状态和排除原因。候选目标没有最终监督资格。当前目标和参考分页在 attempt_materials 中可见，模型选择实际参考组合。

同图跨训练任务允许不同用途；正式测试保留仍生效。未填写正式测试注册表时仅为开发范围。


In [ ]:
print("文章发布：", knowledge_runs)
print("视觉发布：", visual_runs)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, None)
        input_rows = list(input_records(files))
        selected_concepts = design_concepts(input_rows, config)
        target_pool = ExistingImagePool(run, guard, config)
        knowledge = (data.from_iter(lambda: iter(input_rows))
            .map(PrepareTrainingMaterials(files.knowledge_version, target_pool, config, guard, selected_concepts))
            )
        knowledge = files.lance_checkpoint(knowledge, "training_materials")
        scope = knowledge.filter(lambda r: r["authoring_selected"] and r["status"] != "needs_materials")
        if config['training_design'] == 'target_aware':
            target_visits = training_targets(knowledge.take_all(), config)
            progress = loop_progress()
            if not target_visits:
                raise ValueError('No eligible targets; inspect training_materials')
            attempt = training_attempt(target_visits, progress, config)
            task_prefix = attempt['unit_id']
            scope = data.from_iter(lambda: iter([attempt]))
            scope = files.lance_checkpoint(scope, prefix + 'attempt_materials')
        state = files.finish()
else:
    knowledge = data.from_iter(lambda: rows(snapshot_ref(run, "training_materials")))
    scope = (data.from_iter(lambda: rows(snapshot_ref(run, "attempt_materials")))
             if config["training_design"] == "target_aware" else knowledge.filter(lambda r: r["authoring_selected"] and r["status"] != "needs_materials"))


In [ ]:
show_records(knowledge, "training_materials", CASE_ID, audit=SHOW_AUDIT)
if config["training_design"] == "target_aware":
    show_records(scope, "attempt_materials", CASE_ID, audit=SHOW_AUDIT)


## 2. 模型选择目标／参考组合并构题

程序给定一个目标与一批材料，模型选择可用 evidence 并直接出题，文本可选；每次至多一条样本。程序核查近重复及配图依赖，模型核查支持范围和是否退化为完整答案复刻。

输出 `target_candidates`、`evidence`、`reference_selection_reason`、知识应用与任务草案。选中的 evidence 同时绑定这道题实际输入的参考图列表及可选知识文本；后续只校验和审核，不再重新检索选材。

下面保存本次完整 prompt 快照，可直接阅读，无需执行 notebook；紧随的 `show_prompt` 显示当前源文件。


### design_candidates 完整 prompt（本次代码同步快照）

你是V2图像任务的候选设计者。根据本次给定的已发布材料，构造目标明确、指导信号有依据、正误可观察的简单任务。一次完成选择和设计，不先输出另一份知识摘要、发现清单或人工计划。以下要求用于帮助你判断具体候选，不是逐项凑齐的配额；只输出末尾约定的JSON。

训练材料与选图契约（V2 本轮待审核）：

当前训练 policy.one_sample_per_target_visit=true 时，本次已给定唯一候选目标和一批参考候选。围绕这个目标选参考并直接构造至多一条样本，target_candidates 固定为 [1]；不能改选未提供的目标。没有合适组合就返回 insufficient。程序会在样本通过审核后换下一目标；目标池走完可以重复目标，不要求题面不同。样本计数、翻批和循环由程序处理，你不维护计数，也不需要历史题目列表。

- materials 是已发布的可选依据，可能含知识文本、文章配图和独立视觉材料。知识文本不强制引用；纯视觉依据、纯文字依据、图文共同依据均可。不得因没有文章就拒绝有充分视觉支持的任务。
- 候选监督目标来自显式声明的发布源。独立视觉发布经过概念审核，文章携带的原始候选不一定经过同样审核；均不自动取得本题监督资格。先看候选目标可承载什么任务，再结合可用依据设计任务；不能从目标的偶然外观编出知识规则。
- 同一视觉库不预分参考池和目标池。每个 target_reference_options 条目对应一个候选目标；evidence 列出与它不重复、且依赖配图未被排除的可用资料编号，excluded 列出程序排除原因。编号均对应本次输入原始资料号。
- 对每道题先选择 target_candidates，再从对应可用编号中选择 evidence。选多个目标时，evidence 必须同时适用于全部目标；若各目标需要不同参考组合，应拆成不同候选任务。候选目标编号和资料编号属于两个编号空间，不能混用。
- 只选择真正支持本题的依据，不必用完列表，也不强制图文齐全。reference_selection_reason 简短说明所选参考提供什么，以及题目仍需进行什么选择、条件应用或关系迁移。资料号只放 evidence；说明用事实名称，不写重编号后可能失效的资料号。
- 程序只做图片身份与近重复检查。你还须看像素，排除不同图片却直接展示本题完整答案、只需复刻的组合。相同概念、相同必要结构不自动构成泄漏，参考可以展示正确知识。
- 一张图即使在某个选项中是候选目标，仍可用于另一道题的参考；只有选定组合内禁止目标及其近重复充当依据。不得因为它出现在候选目标列表就全局禁用。
- 可用编号为空、关键知识无支持或无法构成知识应用时，返回 insufficient 并说明具体缺口；不能把目标自身充作证明。目标最终合格与否由冻结任务后的独立审核决定。
- 训练任务的 evidence 同时确定本题实际输入的参考图列表与可选知识文本，不只是供出题者看的依据。选择目标、参考与题目是同一次联合设计；后续不得重新检索、替换或补入另一套参考。需要更换材料时应重新设计并审核该任务。图片、知识文本均按实际需要选择，不强制图文齐全。

训练 evidence 将绑定为实际输入；target_aware=true 时，每个候选额外输出 target_candidates 和 reference_selection_reason，例如：
{"target_candidates":[1],"evidence":[2,3],"reference_selection_reason":"参考展示符号形态和连接规则，任务需要把这些规则应用到给定条件分支。"}
这只是字段示例；编号必须来自实际输入。其余 task_type、knowledge_gap、knowledge_application、draft 或编辑需求沿用下文契约。

一、任务目标与本步职责

主线研究任务条件＋知识／视觉参考如何推导出应有的视觉结果。knowledge_application 用简短自然语言写清“给定条件→应用哪条知识→得到什么可见结果”，不要求显式长思维链或多跳数量。优先让任务简单、目标清楚、判据可观察；不需要证明模型内部缺少知识，也不需要把难点包装成多跳推理。

先提出有材料支持的候选，实际难度由后续模型试做判断。明确失败的开发任务可进入训练候选池，最终价值由训练及未训练题目的验证确定。不能预测模型必错，不能要求推理时补充信号立刻改善，也不能因预计training-free收益为零而拒绝题目。单纯照抄题面全部视觉要求的执行题仅是辅助诊断，本批优先保留真实的知识选择、条件应用、关系绑定或迁移；正式测试仍需独立保留，不按方法收益反选或回流训练。

只在policy.task_types允许的轨道内设计，最多policy.max_candidates条，数量是上限，不必用满。候选之间应有实际的知识或应用差异，换名称、配色、构图不自动构成新考点。不按领域、知识类别、应用方式、编辑类型、资料形态、对象数或推理跳数凑题。不因概念常见／冷僻、是实体／非实体或未提供taxonomy就直接取舍。本次只能讨论实际收到的知识，不能声称已探索未提供的领域或全库材料。

本轮开发优先目标：从收到的知识中选择一个清楚可见的核心事实或紧密关联的结构关系，形成能直接试做的题目。一般优先较大的外形、部件组织、明确状态或接触关系；精确计数、微小纹样、遮挡依赖和强制多视图只有在考点不可替代时才采用。完整必要身份仍须成立，但不把同一概念的全部知识都塞进一题。不要为“体现知识”强加功能场景、局部放大图或组合多个主题。材料不足以支持清楚目标时返回不足，不用额外复杂度补难度。

出题criteria将作为评测逐条原样使用的题目级rubrics，不再由评测另行扩写。每条requirement写清可见的成立条件与明确反证，observable_region给出足以观察的位置，allowed_variation保留合法变化；把必须检查的主体身份、核心知识差异和必要任务执行分别表达，避免一句囊括全部事项。判据的观察要求必须在公开题面中交付，不能用隐藏condition代替。判据数量以必要为准，不为了覆盖知识库全部内容而增加。不要预测Qwen或任何模型必错、必因知识改善。

二、证据是什么，能支持到哪里

输入中的知识正文、文章配图与独立发布的视觉材料是本步依据；保留其身份范围、条件、例外、支持限制及来源标识。已发布不等于每个说法都自动正确：若输入内部冲突、缺少决定性前提或图文身份不一致，应保留不足，不用自身记忆、标题、作者旧判断或模型caption补成确定事实。来源标识不等于你已经打开过来源页面。本步不搜索外部知识，也不重新提取上游未发布事实。

- 文字支持其实际陈述的事实、条件和关系。不要把某一地区／时期／版本／状态下的陈述推广为唯一普遍规则，也不要把关联、可能或尝试写成因果、必然或已证实有效。
- 图片必须实际看像素，判断本题所需身份及可见内容，已有来源信息可辅助核对。不要求原始发布页、作者、拍摄记录或非生成证明；图片直链、缺少出处或origin=not_verified本身不是拒绝理由。它可以支持清楚可见的结构、部件、纹样、位置或状态；普通外观照片不能证明内部连接、成分、机制、精确数值或整个类别的一般规律。模型caption只是线索；原来源图注可作为有出处的文字考虑，但不能自动认证图中对象。
- 文字、图片和图文都可以成为有效依据。图片不必含有“绝对无法用文字描述”的知识才有价值；关键是它具体提供了什么、哪部分可见、哪些部分不能确认。多图可分工，一张图也可只支持部分判据。文字充分时不强配图；输入有实质视觉知识时，也不要因文字较易写而忽略它。
- 参考图展示正确知识本来就是知识增强的目的，不因此一律判为泄漏。但本题目标图、近重复答案构图或只需临摹的完整成品，不能伪装成知识应用证据。判断重点是仍需从资料选择、绑定或迁移什么，不要求每题强造新背景。
- 知识参考配图、编辑原图、监督目标是不同角色。输入角色以实际标记为准，不得把参考默认为原图或目标。Benchmark T2I 不需要目标；训练 policy.target_aware=true 时会提供候选监督目标，允许结合它们设计合适任务，但它们不是知识证据，不能从偶然外观编造规律或只看图改写 caption。

三、选择考点时的具体边界

下列六类描述知识内容，可交叉使用，不是难度等级，不要求输出分类字段或覆盖所有类别。

1. 特征与结构：考来源支持的必要身份、形态、部件及组织关系，区分概念特征与单个实例的偶然外观。不能只验局部花纹而遗漏本题身份成立所需的结构；也不为“完整”穷举无关微小细节，或要求照抄单张图的全部外观。
2. 属性与状态：写清阶段、温度、浓度、环境、版本等真正影响结论的条件及可接受范围。不能从照片猜不可见成分或精确数值；题面已给出的颜色不是缺知识的证据，但仍可作为有依据的执行目标。颜色知识可以考，但不要把所有知识都压成颜色映射。
3. 功能与机制：先分清题目要验工作结果、必要连接，还是内部机制本身，证据与图像表达要对应。只看得到结果不能宣称完整机理已验证；考功能也不必强行画全套内部剖面。可靠文字可以支持机制示意，外观相似不能支持机制结论。
4. 过程与变化：区分某阶段长什么样与阶段间如何变化；给定足以选择结果的条件和时刻。不要用含糊的“过一段时间”要求唯一终态，不把可能变化写成必然，也不强制把所有阶段拼在一张图中。单图只验其能表达的阶段或变化关系。
5. 关系与组织：明确关系由哪项知识决定、作用于哪些对象或部件、如何在当前情境观察。普通指定摆放、对象数量多、连线或标签多，都不自动是关系知识。对象身份正确也不等于接触、方向、归属或组织关系正确。
6. 规则与约定：保留来源地位、版本、地区、历史或文化范围、例外及合法变体。不要把一种传统说成唯一答案，也不要为了难倒强模型选择无可靠支持的规定。规则的语义只在有合适图像表达时成为可判要求，不靠堆整套仪式或密集文字假装全部视觉化。

再判断这道题主要怎样使用知识，并在knowledge_application中用简短自然语言说清；不用新增标签字段：

- 直接应用direct：题面点名对象或任务，资料补足题面没写出的特征、结构或规则。说明具体空缺；不为升级难度硬加情境。题面已经列明特征时，仍可考查这些信号是否被正确执行，不因此拒绝。
- 条件应用conditional：条件用来选择适用规则、阶段或版本，而不是仅添一个环境词。不同条件可能得到同一外观，不强制它们必须不同；要解释条件实际决定了什么。当前材料若只给了一条适用事实，不冒称作答者必须从大量冲突规则中检索选择。
- 关系应用relational：把有依据的关系绑定到当前对象、部件、位置或状态，必要时从参考情境迁移。保持必要接触、方向和位置；题面直接指定的摆放本身不能冒充知识推断。
- 组合应用compositional：多条知识共同决定结果，可以是连续推导，也可以是共同约束。简短说明各条为何必要。同一事实拆成几个判据不算组合，多个对象各自应用一条知识只是并列要求；不设最低跳数、领域数或知识条数，不牺牲证据范围强行拼接。

四、题面与判据怎么写

训练候选尤其要把“一个核心信号”落实到范围：先选一个可见差异或一组不可分割的关系，再只保留完成它所需的身份和展示条件。不要把同一对象多个独立部位的外形合并成完整形态考试，也不要因资料介绍了用途就强加整套操作。若另一个部位画法改变而核心事实仍成立，它通常应另出一题，而非追加到本题。主体仍须与所指概念相符；身份核验不等于要求所有鉴别特征在一个视角同时显露。核心在局部时可明确采用局部展示，并保留足以判断所属对象的上下文，不要求无关部位全部入镜。判据可以分条表达同一核心信号及必要前提，但不借拆条增加独立考点。候选设计阶段可据材料共同确定范围；任务与判据冻结后不能为迁就目标放宽要求。

训练数据分支在 policy.target_aware=true 时依据知识、参考和候选目标共同构题；候选目标只帮助判断可监督的结果，不是独立事实证据，仍须后续独立审核。每个候选追加 target_candidates 数组，填所选目标候选编号（从1开始，可多张）；不要把目标编号写入 evidence，evidence 只引用知识资料。未提供目标的 knowledge_first 模式无需该字段。若媒介、背景和姿态不是考点，不将“必须是插画／照片”、特定风格或装饰构图设为硬要求；可以请求清楚表现相应对象、结构或关系的图像，并允许自然照片、科学插图等满足同一可见判据的表示。不得为适配现有目标而删除必要展示条件、降低身份要求或预先假定可找到合格目标。

题面给出任务、适用条件和必要展示要求，可以明确写出有依据的视觉要求，不为制造知识空缺而隐藏必要信息。保留接口字段knowledge_gap，用它说明需要从材料补足或落实到画面的内容；保留必要可观察条件，不将待推导的全部结论预先列进题面；如没有剩余知识应用，不能假称有推理。knowledge_application说明指导信号与可见结果的对应。不输出隐藏评分编号或评测结论，不把完整目标成品伪装成参考信号。

只保留表达考点所需的场景负担：删除一个要求后，若知识仍能完整考查，它通常不应成为额外硬要求。知识本身必需的对象身份、形态、空间分布、接触、方向、状态及必要标签则不能为了好画而删掉。结构或空间表达难，并不证明它与知识无关。

具体取舍：不主动堆实体、细节、前中后景或多人动作；过程、环境、标准语境只保留真正选择知识的条件；必要剖面／视角可以要求，但不附加无关多视图；交互和多实例对比要有知识必要性，不用长链和复杂排版增加难度；光照、时段、运动仅在决定考点时保留，不追加氛围、飞溅和同步阴影等负担。示意图可用必要短标签、箭头与图例，但不能用一段正确文字代替应画出的结构关系。

为核心判据指定可观察部位，保证目标区域、尺度、视角和遮挡要求足以判定；检查题面是否允许一种合法完成却把考点完全藏住的画法。如果存在，应补展示条件或换表达方式，不能等评分时把未写明的要求强加给结果。不可见的内部状态须有可靠依据和明确示意／暴露方式，不能让普通照片承担它无法证明的内容。

每条criterion分别写清可见要求、evidence、observable_region和allowed_variation。身份、必要结构及知识前提应覆盖，合法的朝向、布局、色差、阶段范围或表示法应放开；不要求唯一像素答案，不用无来源的精确数值收窄正确答案。拆条只是便于核验，不增加难度，也不等于已经实施逐项K评分。

五、编辑候选：先给原图需求，看到图后才定稿

编辑只设计一次主修改，优先adjust／replace／add；允许知识必需的连带变化，不把单一目标误认为一定容易执行。edit_intent写要改什么及知识作用，scene_requirements写初始对象／承载场景、状态、锚点可见性、空间和合理保持条件。不要虚构当前图片中的位置或已核验的物种身份。

- adjust改变原主体的属性或状态；若实质是换对象，应按replace思考。replace要验替换后的完整必要身份与原槽位关系，不能只换颜色。add应有明确承载区域或关系锚点，原图可以尚无目标物；不要求它属于目标知识概念。
- action只在行为／用法本身有知识依据、真实原图能清楚承载时选用；不主动叠加精细手指、多关节和多工具同步。remove要说明删什么为何依赖知识，不能要求恢复不可见背景的唯一真相。background要允许环境变化必需的前景联动，不能要求不可能的全像素保持。style须有来源支持的具体特征及合法变体，不能只考“像某风格”。后三类降低优先级，不为覆盖而选。
- 不使用extract或compose编辑；一次主操作仍可应用多条知识。不得要求生成、自制或补画编辑底图。

search_queries给1–3条简短查询，检索的是满足初始状态的现成场景，不是最终答案图。保留真正必要的身份限定，使用已给出的名称／别名，可组合中文和合适英文描述；不要凭空发明别名。按场景需要允许跨概念搜索，例如添加物体应搜承载场景，替换可搜原槽位中的其他对象；涉及物种或特定文物自身状态变化时，则不能随意拿相似对象顶替。区分必要初态与审美偏好，不把原图条件堆成几乎不可能的组合，也不能为提高命中率放宽关键身份。材料是否存在由后续检索核验，本步不声称搜过或找到。

六、输出约定

只输出业务JSON，外层result包装由调用模板约定。不要新增ID、URL、哈希、分类报表、难度分数、combo_type、premise_types或配额字段。说明保持简短，重点是知识空缺、必要关系、实际依据和限制，不复述逐项思考过程。

候选evidence及criterion.evidence均引用本次知识资料的编号，从1开始；候选evidence覆盖该题全部依据，若使用配图也引用其资料号。编号只写在evidence数组中；instruction、knowledge_application、knowledge_gap、requirement等自然语言用实际事实名称表达，避免资料选取后重编号留下“资料7／8”这样的悬空指代。不编造资料或借未经支持的常识补判据。

T2I直接给完整draft；共享knowledge_application只在候选顶层写一次，draft不重复。edit_type=null、anchor=""、preserve=[]。编辑候选的draft=null，具体题面与锚点等拿到真实原图后再定稿。不因当前未给编辑原图就自动认定知识无效；若核心知识本身不足以形成任何允许的可判任务，则返回insufficient并说清具体缺口。

下列示例省略可选字段；当 policy.target_aware=true 时，每个 T2I 或编辑候选必须另含 target_candidates:[1] 和 reference_selection_reason（可列多个本次候选目标编号），与 evidence 的知识编号严格分开。无此策略时不填 target_candidates。

T2I格式：
{"status":"ok","candidates":[{"task_type":"t2i","evidence":[1],"knowledge_gap":"需要补足或落实到画面的指导信息","knowledge_application":"主要应用方式及条件、知识、可见结果的关系","draft":{"status":"ok","instruction":"给作答模型的题面","condition":"实际适用条件","criteria":[{"requirement":"可见要求","evidence":[1],"observable_region":"在哪里、怎样观察","allowed_variation":"合法变化与支持限制"}],"edit_type":null,"anchor":"","preserve":[]}}]}

编辑候选格式（置于同一candidates数组）：
{"task_type":"edit","evidence":[1],"knowledge_gap":"需要补足或落实到画面的指导信息","knowledge_application":"主要应用方式及知识决定的结果","edit_intent":"一次主修改及其知识目的","scene_requirements":"必要初态、身份和可见锚点要求","search_queries":["简短初始场景查询"],"draft":null}

无合格候选：{"status":"insufficient","reason":"具体的证据、适用范围或可判任务缺口；不要泛称整个领域不适合"}。


提交前逐条尝试构造“符合公开题面却被本判据判错”的合法画法。特别是透视、反射、遮挡及相对位置：物体在场不等于进入镜头或反射区域，观察者面向某处也不保证指定对象可见。若判据要求辨认某个对象，须在公开题面明确确保相关位置和视野；若所需前提过多，则缩小题目，只考能直接观察的一条事实。必要的展示条件及有依据的视觉要求可以写入题面；不把隐藏评分字段和目标成品混入模型输入。对每条判据做完这一反例检查后再提交。


In [ ]:
show_prompt("design_candidates")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, "attempt_materials" if task_prefix else "training_materials")
        check_step_order(files, "attempt_materials" if task_prefix else "training_materials")
        designed = (scope
            .map(partial(prepare_design, run=run, pack=pack))
            .map_prompt_async("design_candidates", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="design_candidates_result", call_output="design_candidates_call", error_output="design_candidates_error",
                when=lambda r: r["status"] == "knowledge_available", concurrency=1, queue_depth=1)
            .map(partial(apply_design, run=run))
)
        designed = files.lance_checkpoint(designed, prefix + "design", extra=prompt_responses(run, "design_candidates", task_prefix))
        state = files.finish()
else:
    designed = data.from_iter(lambda: rows(snapshot_ref(run, "design")))


In [ ]:
show_records(designed, "design", CASE_ID, audit=SHOW_AUDIT)


## 3. 展开候选与绑定证据

| 项目 | 说明 |
|---|---|
| 实际算子 | `flat_map(ExpandCandidates)` |
| 输入与输出 | 模型候选列表 → 每题一行 |
| 为什么做／边界 | 这一步不调用模型。校验引用、保留必需配图并重编号；派生类型、划分和ID。T2I已有草稿，edit等待原图，坏候选保留原因。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(designed, "design", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'design')
        candidates = (designed.flat_map(ExpandCandidates(guard, config))
)
        candidates = files.lance_checkpoint(candidates, prefix + "candidates")
        state = files.finish()
else:
    candidates = data.from_iter(lambda: rows(snapshot_ref(run, "candidates")))


In [ ]:
show_records(candidates, "candidates", CASE_ID, audit=SHOW_AUDIT)


## 4. 编辑：按初始场景搜本地图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_async(SearchLocalScenes)` |
| 输入与输出 | 编辑原图需求及查询词＋全库图片清单 → 候选原图 |
| 为什么做／边界 | 跨概念扫描caption／标题／已有概念元数据，排序后只读取有界候选像素。不以当前知识概念限制来源；例如添加对象可找尚无该对象的承载场景。排除参考近重复及测试保留，不把候选当合格原图。T2I跳过。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(candidates, "candidates", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'candidates')
        scene_search = SceneSearch(run, knowledge.flat_map(knowledge_items).take_all(), guard, config)
        edit_candidates = (candidates.map_async(SearchLocalScenes(scene_search), concurrency=1)
)
        edit_candidates = files.lance_checkpoint(edit_candidates, prefix + "edit_search")
        state = files.finish()
else:
    edit_candidates = data.from_iter(lambda: rows(snapshot_ref(run, "edit_search")))


In [ ]:
show_records(edit_candidates, "edit_search", CASE_ID, audit=SHOW_AUDIT)


## 5. 仅编辑：核验原图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("select_edit_source")` |
| 输入与输出 | 知识考点、最终知识、候选原图像素与来源 → 合格原图或needs_edit_source |
| 为什么做／边界 | 这个额外输入只属于编辑场景，不是知识。实际核验来源、非生成性质、可见锚点、可实施的一次主操作；不能根据图片回头编造知识。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(edit_candidates, "edit_search", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("select_edit_source")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_search')
        source_ready = (edit_candidates
            .map(partial(prepare_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_result", call_output="select_edit_source_call", error_output="select_edit_source_error",
                when=lambda r: r["status"] == "edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_edit_source, run=run))
)
        source_ready = files.lance_checkpoint(source_ready, prefix + "edit_source", extra=prompt_responses(run, "select_edit_source", task_prefix))
        state = files.finish()
else:
    source_ready = data.from_iter(lambda: rows(snapshot_ref(run, "edit_source")))


In [ ]:
show_records(source_ready, "edit_source", CASE_ID, audit=SHOW_AUDIT)


## 6. 编辑：必要时外部补搜

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_async(SearchExternalScenes)` |
| 输入与输出 | 本地无候选或像素审核拒绝 → 外部候选或可追溯缺口 |
| 为什么做／边界 | 默认Wikimedia Commons，可配置SearxNG图片搜索。使用模型已给出的初始场景查询；搜索数、下载数与字节数有界。来源、许可、实际像素、错误均冻结；续跑不重复请求。不是生成底图。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(source_ready, "edit_source", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_source')
        external_candidates = (source_ready.map_async(SearchExternalScenes(scene_search), concurrency=1)
)
        external_candidates = files.lance_checkpoint(external_candidates, prefix + "edit_external_search")
        state = files.finish()
else:
    external_candidates = data.from_iter(lambda: rows(snapshot_ref(run, "edit_external_search")))


In [ ]:
show_records(external_candidates, "edit_external_search", CASE_ID, audit=SHOW_AUDIT)


## 7. 编辑：核验外部原图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("select_edit_source_external")` |
| 输入与输出 | 外部候选像素及来源 → 可用原图或缺口 |
| 为什么做／边界 | 与本地原图共用同一业务prompt和准入要求；独立命名只区分不同请求输入与断点。看不清锚点、来源不足或任务不可实施就保留缺口。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(external_candidates, "edit_external_search", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("select_edit_source_external")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_external_search')
        all_sources = (external_candidates
            .map(partial(prepare_external_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source_external", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_external_result", call_output="select_edit_source_external_call", error_output="select_edit_source_external_error",
                when=lambda r: r["status"] == "external_edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_external_edit_source, run=run))
)
        all_sources = files.lance_checkpoint(all_sources, prefix + "edit_external_source", extra=prompt_responses(run, "select_edit_source_external", task_prefix))
        state = files.finish()
else:
    all_sources = data.from_iter(lambda: rows(snapshot_ref(run, "edit_external_source")))


In [ ]:
show_records(all_sources, "edit_external_source", CASE_ID, audit=SHOW_AUDIT)


## 8. 编辑：据真实原图定稿

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("construct")` |
| 输入与输出 | 候选设计＋选定原图像素 → 具体题面和判据 |
| 为什么做／边界 | 只在编辑原图审核通过后执行，绑定实际锚点、一次主操作和必要保持。T2I沿用design直接给出的草稿，不再重复构题。不能为迁就原图改知识或看目标倒写题。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(all_sources, "edit_external_source", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("construct")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_external_source')
        constructed = (all_sources
            .map(partial(prepare_construct, run=run, pack=pack))
            .map_prompt_async("construct", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="construct_result", call_output="construct_call", error_output="construct_error",
                when=lambda r: r["status"] == "selected", concurrency=1, queue_depth=1)
            .map(partial(apply_construct, run=run))
)
        constructed = files.lance_checkpoint(constructed, prefix + "construct", extra=prompt_responses(run, "construct", task_prefix))
        state = files.finish()
else:
    constructed = data.from_iter(lambda: rows(snapshot_ref(run, "construct")))


In [ ]:
show_records(constructed, "construct", CASE_ID, audit=SHOW_AUDIT)


## 9. 校验任务契约

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(ValidateTask)` |
| 输入与输出 | 草稿和材料编号 → 绑定稳定知识ID的判据及冻结任务哈希 |
| 为什么做／边界 | 检查字段、来源编号、编辑类型、隔离与禁止的旧配额字段。程序通过只说明契约合法，不证明知识语义正确。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(constructed, "construct", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'construct')
        validated = (constructed.map(ValidateTask(guard))
)
        validated = files.lance_checkpoint(validated, prefix + "validate")
        state = files.finish()
else:
    validated = data.from_iter(lambda: rows(snapshot_ref(run, "validate")))


In [ ]:
show_records(validated, "validate", CASE_ID, audit=SHOW_AUDIT)


## 10. 审核任务质量

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("review_task")` |
| 输入与输出 | 草稿、全部构题证据、编辑原图 → 五项语义审核 |
| 为什么做／边界 | 检查来源支持、知识必要、可观察、不泄漏、保持合理。新请求无作者聊天；通过仍是reviewed_candidate_not_golden，不授予人工golden。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(validated, "validate", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("review_task")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'validate')
        reviewed = (validated
            .map(partial(prepare_task_review, run=run, pack=pack))
            .map_prompt_async("review_task", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="review_task_result", call_output="review_task_call", error_output="review_task_error",
                when=lambda r: r["status"] == "valid_task", concurrency=1, queue_depth=1)
            .map(partial(apply_task_review, run=run))
)
        reviewed = files.lance_checkpoint(reviewed, prefix + "review", extra=prompt_responses(run, "review_task", task_prefix))
        state = files.finish()
else:
    reviewed = data.from_iter(lambda: rows(snapshot_ref(run, "review")))


In [ ]:
show_records(reviewed, "review", CASE_ID, audit=SHOW_AUDIT)


## 11. 校验并组装已选训练输入

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(BindTrainingInputs)` |
| 输入与输出 | 构题时绑定的参考／可选文本 → 校验并组装实际输入 |
| 为什么做／边界 | 核对所选材料内容、顺序及来源绑定未变，覆盖判据且不与目标／编辑初态重复。通过后原样组装训练输入；失败则保留缺口，不检索替换或自动补材料。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(reviewed, "review", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'review')
        bound_inputs = (reviewed.map(BindTrainingInputs(guard))
)
        bound_inputs = files.lance_checkpoint(bound_inputs, prefix + "bind_inputs")
        state = files.finish()
else:
    bound_inputs = data.from_iter(lambda: rows(snapshot_ref(run, "bind_inputs")))


In [ ]:
show_records(bound_inputs, "bind_inputs", CASE_ID, audit=SHOW_AUDIT)


## 12. 寻找监督目标

| 项目 | 说明 |
|---|---|
| 实际算子 | `flat_map(SelectTargetCandidate.expand)` |
| 输入与输出 | 冻结题面与声明的素材来源 → 查找与已绑定参考兼容的目标、待验目标或needs_target |
| 为什么做／边界 | 优先验证构题时选中的候选目标，也可从声明来源补选；排除本题构题依据及已绑定训练参考的近重复。冻结后不能为迁就目标改题；不凭caption或路径当作合格目标。没有候选就未完成。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(bound_inputs, "bind_inputs", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'bind_inputs')
        targets = (bound_inputs.flat_map(SelectTargetCandidate(guard, target_pool, limit=1 if task_prefix else config.get("target_candidates_per_task", 1)).expand)
)
        targets = files.lance_checkpoint(targets, prefix + "targets")
        state = files.finish()
else:
    targets = data.from_iter(lambda: rows(snapshot_ref(run, "targets")))


In [ ]:
show_records(targets, "targets", CASE_ID, audit=SHOW_AUDIT)


## 13. 核验训练目标

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("review_target")` |
| 输入与输出 | 冻结任务、依据、原图、候选目标及作答材料 → 目标审核 |
| 为什么做／边界 | 检查全部判据、画质、真实编辑前后对应及保持。仅通过才training_ready；无合格目标不能导出完成训练样本，也不自动生成底图补缺口。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(targets, "targets", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("review_target")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'targets')
        target_reviewed = (targets
            .map(partial(prepare_target_review, run=run, pack=pack))
            .map_prompt_async("review_target", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="review_target_result", call_output="review_target_call", error_output="review_target_error",
                when=lambda r: r["status"] == "target_attached", concurrency=1, queue_depth=1)
            .map(partial(apply_target_review, run=run))
)
        target_reviewed = files.lance_checkpoint(target_reviewed, prefix + "review_targets", extra=prompt_responses(run, "review_target", task_prefix))
        state = files.finish()
else:
    target_reviewed = data.from_iter(lambda: rows(snapshot_ref(run, "review_targets")))


In [ ]:
show_records(target_reviewed, "review_targets", CASE_ID, audit=SHOW_AUDIT)


## 14. 导出并保留缺口

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(export_record) → filter` |
| 输入与输出 | 任务及各阶段审核 → ready／incomplete |
| 为什么做／边界 | Benchmark输出题面和判据；训练输出知识／原图输入与仅在目标计算loss的序列。未采用、失败、待响应和缺目标保留。逐项K评分与BAGEL读取器不属于这次修改。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(target_reviewed, "review_targets", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'review_targets')
        exported = (target_reviewed.map(export_record)
)
        exported = files.lance_checkpoint(exported, prefix + "export")
        ready = (exported.filter(lambda r: r["export_ready"])
)
        ready = files.lance_checkpoint(ready, prefix + "ready")
        incomplete = (exported.filter(lambda r: not r["export_ready"])
)
        incomplete = files.lance_checkpoint(incomplete, prefix + "incomplete")
        state = files.finish()
else:
    exported = data.from_iter(lambda: rows(snapshot_ref(run, "export")))


In [ ]:
show_records(exported, "export", CASE_ID, audit=SHOW_AUDIT)


## 全链路概览

续跑时从初始化开始，依次执行；已有checkpoint自动复用。新增offline响应仅使对应模型步及其下游产生新revision，旧结果原样保留。

In [ ]:
show_summary(run)


## 按总条数继续循环

只在 MODE="execute" 时执行。复用上方已完成的同一 run，直到达到合格样本数、等待响应或耗尽预算。当前默认仍为只读；本次没有运行正式数据生产。


In [ ]:
if MODE == "execute":
    import asyncio
    from curation.training.runtime import load_pipeline
    state = await asyncio.to_thread(load_pipeline("training"), run, knowledge_runs, config,
                                    visual_runs=visual_runs)
    print(state.get("training_progress", {}))
else:
    print("只读模式：循环未启动。")
